# "THE POUR IS RIGHT" - Wine Pricer Capstone

## Week 8, an extra: the planning agent in LangGraph

Ed's course builds agents by hand, so you can see how they work underneath. This notebook rebuilds the same planning agent with **LangGraph**, a framework for agents that is widely used in industry. The job is unchanged: find deals, price them, and alert you about the best one. Only the way it is built differs, so you can compare the two directly.

| Version | Who decides what happens next |
|---|---|
| `PlanningAgent` (day 4) | Plain Python, in a fixed order |
| `AutonomousPlanningAgent` (day 4) | GPT-5.1, choosing which tool to call |
| `LangGraphPlanningAgent` (this notebook) | A graph you design: steps, and rules for moving between them |

## The ideas

LangGraph describes a program as a **graph**:

- **State**: a dictionary that holds everything the program knows so far: the deals, the estimates, the best deal. It is passed from step to step.
- **Node**: one step. A Python function that takes the state and returns the parts of it that it changes.
- **Edge**: "after this node, go to that one".
- **Conditional edge**: a function that looks at the state and chooses the next node. This is how the graph makes decisions.
- **Send**: starts several copies of a node, one per item. We use it to price each deal in its own branch, which is called fan-out.
- **Reducer**: says how to combine the results when several branches update the same part of the state. Ours joins the branches' lists into one.
- **Checkpointer**: saves the state after every step. That is what makes the next one possible.
- **Interrupt**: pauses the graph partway, waits for a person, then carries on from where it stopped. This is called human in the loop.

Our graph:

```
START -> scan --(no deals)----------------------------------> END
           |
           +--> price_deal  (one branch per deal, via Send)
                     |
                     v
                  decide --(not worth an alert)--> END
                     |
                     +--> notify (optional pause for approval) --> END
```

The code is in `agents/langgraph_planning_agent.py`. Read it alongside this notebook: each node is a short method.

## Costs

The practice run uses pretend agents and is free. Each real run costs about 5 cents plus a little Modal credit, takes about a minute and a half, and can send one Telegram message.

In [ ]:
import logging
from dotenv import load_dotenv
from qdrant_client import QdrantClient
from agents.langgraph_planning_agent import LangGraphPlanningAgent
from agents.scanner_agent import ScannerAgent

load_dotenv(override=True)

# Practice with pretend agents

As on day 4, we practise for free first. These three classes stand in for the scanner, the ensemble and the messenger. The pretend ensemble values the Decades Cabernet at \$140 and every other wine at \$20, so exactly one deal should be worth an alert.

In [ ]:
class PretendScanner:
    def scan(self, memory=[]):
        print("Pretend scanner: returning the three test deals")
        return ScannerAgent.test_scan(None)

class PretendEnsemble:
    def price(self, description):
        estimate = 140.0 if "Decades" in description else 20.0
        print(f"Pretend ensemble: {description[:40]}... is worth ${estimate}")
        return estimate

class PretendMessenger:
    def alert(self, opportunity):
        print(f"Pretend messenger: would send an alert about {opportunity.deal.url}")

In [ ]:
practice = LangGraphPlanningAgent.from_agents(PretendScanner(), PretendEnsemble(), PretendMessenger())

## Look at the graph

LangGraph can describe the graph it built. Solid arrows are fixed edges; dotted arrows are the choices made by conditional edges.

The output is in Mermaid, a text format for diagrams. To see it as a picture, paste it into https://mermaid.live

In [ ]:
print(practice.graph.get_graph().draw_mermaid())

## Run it and watch each step

`stream` runs the graph and reports each node as it finishes. Look for three separate `price_deal` updates: that is the fan-out, one branch per deal.

In [ ]:
config = {"configurable": {"thread_id": "practice-1"}}

for update in practice.graph.stream({"memory": [], "opportunities": []}, config, stream_mode="updates"):
    for node, changes in update.items():
        print(f"--- {node} changed: {list(changes) if changes else 'nothing'}")

## Look inside the saved state

The checkpointer kept the final state of that run. `get_state` reads it back: here are the three priced deals, gathered from the three branches by the reducer, and the one chosen as best.

In [ ]:
state = practice.graph.get_state(config).values

for opportunity in state["opportunities"]:
    print(f"${opportunity.deal.price:>6.2f} offered, ${opportunity.estimate:>6.2f} estimated, discount ${opportunity.discount:>6.2f}")

print(f"\nBest: {state['best'].deal.url}, worth an alert: {state['worth_alerting']}, alert sent: {state['alerted']}")

## Pause for approval

With `require_approval=True`, the `notify` node calls `interrupt()` before sending anything. The whole graph stops there and hands back a question. Nothing more happens until it is resumed with an answer.

`start()` runs until the pause. `resume()` continues with your answer.

In [ ]:
practice = LangGraphPlanningAgent.from_agents(PretendScanner(), PretendEnsemble(), PretendMessenger(), require_approval=True)

config, question = practice.start()
question

In [ ]:
# The graph is waiting at this node

practice.graph.get_state(config).next

In [ ]:
# Change to False to reject the alert

approved = True

practice.resume(config, approved)

One thing to know about `interrupt()`: when the graph resumes, the paused node runs again **from its beginning**, and this time `interrupt()` returns your answer instead of pausing. So any code placed before `interrupt()` in a node runs twice. That is why our `notify` node does nothing before asking, and only sends the alert after the answer comes back.

# Now with the real agents

The same graph, built with the real ScannerAgent, EnsembleAgent and MessagingAgent.

**Pricing runs one deal at a time.** The fan-out could price all five deals at once, but each simultaneous call to the fine-tuned model would make Modal start another machine, with its own one-minute start-up and cost. So `max_concurrency=1` keeps the branches running in turn. With services that handle many requests at once, you would raise it and the pricing would run in parallel.

**The estimates vary between runs.** The scanner words each wine's description a little differently every time, and the estimate follows the wording. In two of my test runs, the same Decades Cabernet was valued at \$141 and at \$94.

In [ ]:
root = logging.getLogger()
root.setLevel(logging.INFO)

In [ ]:
client = QdrantClient(path="wines_vectorstore")

## A run with your approval

The next cell runs the real graph until it pauses, which takes about a minute and a half. If no deal is worth an alert, it finishes without pausing and `question` is `None`.

In [ ]:
agent = LangGraphPlanningAgent(client, require_approval=True)

config, question = agent.start()
question

If you approve, the alert goes to your Telegram.

In [ ]:
approved = True

agent.resume(config, approved) if question else None

## A run without stopping

`plan()` runs the whole graph without pausing, with the same interface as the other two planners. This is how the app will run it tomorrow.

In [ ]:
agent = LangGraphPlanningAgent(client)
agent.plan()

## Before you leave

In [ ]:
client.close()

# For your CV

What this notebook demonstrates, in terms you can defend in an interview:

- Built an agent workflow in LangGraph with typed state, conditional routing, and parallel fan-out over items using `Send` with a list reducer
- Added human-in-the-loop approval with `interrupt()` and a checkpointer, so a run pauses before an action and resumes with the user's decision
- Integrated the graph with a RAG pricing ensemble (Qdrant, Claude, a QLoRA fine-tuned model served on Modal) and Telegram notifications
- Compared it against a hand-written agent loop and a fixed workflow doing the same job